# 00 — Getting the Data

Downloads real CFPB complaint data. Pick one option by setting `MODE` below:

- `"api"` — download a date range of complaints that have narratives (quick, recommended)
- `"bulk"` — download the whole dataset, then keep only rows with a narrative (big and slow)
- `"synthetic"` — make a small fake file for testing when you are offline

You already have the March 2023 file, so you only need this notebook when you want more data.

In [1]:
import os
import pandas as pd
import requests

# Work from the project root so the "data/..." paths below are correct.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("data/raw", exist_ok=True)

MODE = "api"              # change to "bulk" or "synthetic" if you want
DATE_MIN = "2023-01-01"
DATE_MAX = "2023-06-30"

In [2]:
# Option 1 — API download: a date range, narratives only.
if MODE == "api":
    url = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
    settings = {
        "format": "csv",
        "has_narrative": "true",
        "date_received_min": DATE_MIN,
        "date_received_max": DATE_MAX,
        "field": "all",
        "no_aggs": "true",
    }
    print("Downloading... this can take a minute.")
    answer = requests.get(url, params=settings, timeout=120)
    answer.raise_for_status()

    out_file = "data/raw/cfpb_" + DATE_MIN + "_to_" + DATE_MAX + ".csv"
    with open(out_file, "wb") as f:
        f.write(answer.content)
    print("Saved", out_file)

Downloading... this can take a minute.
Saved data/raw/cfpb_2023-01-01_to_2023-06-30.csv


In [8]:
df = pd.read_csv(out_file)
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-03-11T14:30:23.000Z,"Credit reporting, credit repair services, or o...",Other personal consumer report,Incorrect information on your report,Personal information incorrect,I have written to Transunion twice in regards ...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",GA,30120,NaN,Web,2023-03-11T14:45:02.000Z,Closed with non-monetary relief,Yes,6681450
1,2023-03-11T18:06:41.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Getting a credit card,Sent card you never applied for,SNYCB / Paypal credit card They opened a credi...,Company has responded to the consumer and the ...,SYNCHRONY FINANCIAL,NV,89048,NaN,Web,2023-04-07T17:24:15.000Z,Closed with non-monetary relief,Yes,6680781
2,2023-03-11T14:34:52.000Z,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Personal information incorrect,FTC Report Number : XXXX I am writing to infor...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",MD,21045,Servicemember,Web,2023-03-11T15:37:47.000Z,Closed with non-monetary relief,Yes,6681443
3,2023-03-11T14:45:26.000Z,Debt collection,Other debt,Attempts to collect debt not owed,Debt was result of identity theft,Running my credit report I notice that I had a...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IL,60473,NaN,Web,2023-03-11T14:48:03.000Z,Closed with non-monetary relief,Yes,6681447
4,2023-03-11T14:34:35.000Z,Debt collection,Other debt,Attempts to collect debt not owed,Debt was result of identity theft,Running my credit report I notice that I had a...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IL,60473,NaN,Web,2023-03-11T14:37:26.000Z,Closed with non-monetary relief,Yes,6681442


In [ ]:
# Option 2 — Bulk download: the full file (several GB), then keep only rows with a narrative.
if MODE == "bulk":
    url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
    zip_file = "data/raw/complaints.csv.zip"

    print("Downloading the full file... this is large and slow.")
    # Download in 1 MB pieces so we don't load gigabytes into memory at once.
    with requests.get(url, stream=True, timeout=600) as answer:
        answer.raise_for_status()
        with open(zip_file, "wb") as f:
            for piece in answer.iter_content(chunk_size=1024 * 1024):
                f.write(piece)

    # The file is huge, so read it in chunks and keep only rows that have a narrative.
    out_file = "data/raw/cfpb_all_narratives.csv"
    first_chunk = True
    for chunk in pd.read_csv(zip_file, chunksize=200000, low_memory=False):
        chunk = chunk.dropna(subset=["Consumer complaint narrative"])
        chunk.to_csv(out_file, mode="w" if first_chunk else "a", header=first_chunk, index=False)
        first_chunk = False
    print("Saved", out_file)

In [ ]:
# Option 3 — Synthetic: a small fake file for offline testing. NOT real; never report results from it.
if MODE == "synthetic":
    import random
    random.seed(42)

    products = ["Credit reporting", "Debt collection", "Mortgage", "Credit card", "Checking or savings account"]
    responses = ["Closed with explanation", "Closed with non-monetary relief", "Closed with monetary relief"]
    templates = [
        "There is an account on my report opened on XX/XX/XXXX that I do not recognize.",
        "I was charged a fee of $XXXX that nobody explained to me.",
        "The collector keeps calling about a debt I already paid.",
        "My payment was applied to the wrong account and I got a late fee.",
    ]

    rows = []
    for i in range(2000):
        rows.append({
            "Date received": "2023-0" + str(random.randint(1, 6)) + "-15",
            "Product": random.choice(products),
            "Sub-product": "",
            "Issue": "Incorrect information on your report",
            "Sub-issue": "",
            "Consumer complaint narrative": random.choice(templates),
            "Company public response": "",
            "Company": random.choice(["EQUIFAX, INC.", "WELLS FARGO & COMPANY", "JPMORGAN CHASE & CO."]),
            "State": random.choice(["CA", "TX", "NY", "FL"]),
            "ZIP code": "10001",
            "Tags": "",
            "Submitted via": "Web",
            "Date sent to company": "2023-01-16",
            "Company response to consumer": random.choice(responses),
            "Timely response?": random.choice(["Yes", "Yes", "No"]),
            "Complaint ID": 9000000 + i,
        })

    fake = pd.DataFrame(rows)
    fake.to_csv("data/raw/synthetic_sample.csv", index=False)
    print("Saved data/raw/synthetic_sample.csv with", len(fake), "fake rows")